In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
import os  # Import the os module

# 1. Load and Preprocess the Data
def load_and_preprocess_data(file_path):
    """
    Loads the dataset, preprocesses it, and prepares it for model training.

    Args:
        file_path (str): Path to the CSV file.

    Returns:
        tuple: (train_sentences, train_tags, val_sentences, val_tags,
               test_sentences, test_tags, word_to_idx, tag_to_idx)
    """
    # Check if the file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found at: {file_path}.  Please ensure the file path is correct.")
    
    data = pd.read_csv(file_path, encoding="latin1")
    data = data.fillna(method="ffill")  # Fill missing values

    # Extract words, tags, and sentence numbers
    words = list(data["Word"].values)
    tags = list(data["Tag"].values)
    sentences = list(data["Sentence #"].values)

    # Group data by sentence
    sentence_data = [(s, w, t) for s, w, t in zip(sentences, words, tags)]
    sentence_grouped = {}
    for s, w, t in sentence_data:
        if s not in sentence_grouped:
            sentence_grouped[s] = []
        sentence_grouped[s].append((w.lower(), t))  # Convert words to lowercase

    # Create lists of sentences and tags
    sentences = [ [word for word, tag in sentence_grouped[s]] for s in sentence_grouped.keys()]
    tags = [ [tag for word, tag in sentence_grouped[s]] for s in sentence_grouped.keys()]
    
    # Create word and tag vocabularies
    word_to_idx = {w: i + 2 for i, w in enumerate(set([w for s in sentences for w in s]))}
    word_to_idx["PAD"] = 0  # Add padding token
    word_to_idx["UNK"] = 1  # Add unknown token
    tag_to_idx = {t: i + 1 for i, t in enumerate(set([t for s in tags for t in s]))}
    tag_to_idx["PAD"] = 0  # Add padding token

    # Convert sentences and tags to numerical sequences
    X = [[word_to_idx.get(w, word_to_idx["UNK"]) for w in s] for s in sentences]
    y = [[tag_to_idx[t] for t in ts] for ts in tags]

    # Split data into train, validation, and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

    return X_train, y_train, X_val, y_val, X_test, y_test, word_to_idx, tag_to_idx

# 2. Baseline Model: Most Frequent Tag
def baseline_model(sentences, tags, tag_to_idx):
    """
    Predicts the most frequent tag for each word in the given sentences.

    Args:
        sentences (list): List of sentences (list of words).
        tags (list): List of corresponding tags.
        tag_to_idx (dict): Dictionary mapping tags to indices.

    Returns:
        list: List of predicted tags for each sentence.
    """
    # Flatten the tags list, handling potential numpy arrays
    all_tags = []
    for tag_seq in tags:
        if isinstance(tag_seq, np.ndarray):
            # If it's a NumPy array (from one-hot encoding), get the tag indices
            all_tags.extend(np.argmax(tag_seq, axis=-1).tolist())  # Convert to list
        else:
            # Otherwise, assume it's a list of tag indices
            all_tags.extend(tag_seq)

    # Count tag frequencies, excluding the padding tag
    tag_counts = {}
    for tag in all_tags:
        if tag != 0:  # Exclude padding tag index (0)
            tag_counts[tag] = tag_counts.get(tag, 0) + 1

    # Find the most frequent tag (excluding padding)
    most_frequent_tag = max(tag_counts, key=tag_counts.get)
    
    # Check if most_frequent_tag is in tag_to_idx
    if most_frequent_tag not in tag_to_idx:
        # Handle the case where the most frequent tag is not in tag_to_idx
        # This can happen if the most frequent tag in the test set was rare or unseen in training.
        print(f"Warning: Most frequent tag {most_frequent_tag} not found in tag_to_idx.  Defaulting to padding tag.")
        most_frequent_tag_idx = tag_to_idx['PAD']  # Default to padding tag
    else:
        most_frequent_tag_idx = tag_to_idx[most_frequent_tag]
        
    # Predict the most frequent tag for each word in each sentence
    predicted_tags = [[most_frequent_tag_idx] * len(sentence) for sentence in sentences]
    return predicted_tags

# 3. Evaluation Function
def evaluate_model(y_true, y_pred, idx_to_tag, pad_idx=0):
    """
    Evaluates the model's performance using precision, recall, F1-score, and accuracy.

    Args:
        y_true (list): List of true tag sequences.
        y_pred (list): List of predicted tag sequences.
        idx_to_tag (dict): Dictionary mapping tag indices to tags.
        pad_idx (int, optional): Index of the padding tag. Defaults to 0.

    Returns:
        dict: Dictionary containing evaluation metrics.
    """
    # Remove padding from true and predicted tags
    def remove_padding(seq, pad_idx):
        cleaned_seq = []
        for idx in seq:
            if isinstance(idx, (np.ndarray, list)):
                if len(idx) > 0:
                    first_element = idx[0]
                    if isinstance(first_element, np.ndarray):
                         if first_element.size > 0 and first_element[0] != pad_idx:
                            cleaned_seq.append(first_element.item())
                    elif first_element != pad_idx:
                         cleaned_seq.append(first_element)
            elif idx != pad_idx:
                cleaned_seq.append(idx)
        return cleaned_seq

    true_tags = [remove_padding(seq, pad_idx) for seq in y_true]
    pred_tags = [remove_padding(seq, pad_idx) for seq in y_pred]

    # Flatten the lists for scikit-learn metrics
    y_true_flat = [tag for tag_list in true_tags for tag in tag_list]
    y_pred_flat = [tag for tag_list in pred_tags for tag in tag_list]
    
    # Check for empty lists before proceeding
    if not y_true_flat or not y_pred_flat:
        print("Warning: One or both of the input lists to evaluate_model are empty. Returning default metrics.")
        return {
            "overall_precision": 0.0,
            "overall_recall": 0.0,
            "overall_f1": 0.0,
            "overall_accuracy": 0.0,
            "classification_report": "No data"
        }
    
    # Calculate precision, recall, F1-score, and support
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_flat, y_pred_flat, average=None, zero_division=0)
    report = classification_report(y_true_flat, y_pred_flat, zero_division=0)
    
    # Calculate overall metrics
    overall_precision, overall_recall, overall_f1, _ = precision_recall_fscore_support(y_true_flat, y_pred_flat, average='weighted', zero_division=0)
    overall_accuracy = sum(t == p for t, p in zip(y_true_flat, y_pred_flat)) / len(y_true_flat)

    return {
        "overall_precision": overall_precision,
        "overall_recall": overall_recall,
        "overall_f1": overall_f1,
        "overall_accuracy": overall_accuracy,
        "classification_report": report
    }

# 4. LSTM Model
def build_lstm_model(vocab_size, num_tags, embedding_dim, lstm_units, sequence_length):
    """
    Builds an LSTM-based neural network model for NER.

    Args:
        vocab_size (int): Size of the word vocabulary.
        num_tags (int): Number of distinct NER tags.
        embedding_dim (int): Dimensionality of word embeddings.
        lstm_units (int): Number of units in the LSTM layer.
        sequence_length (int): Maximum sequence length.

    Returns:
        tf.keras.Model: The compiled LSTM model.
    """
    model = Sequential()
    model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=sequence_length))
    model.add(LSTM(units=lstm_units, return_sequences=True))
    model.add(TimeDistributed(Dense(num_tags, activation="softmax")))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    return model


In [4]:
file_path = "ner_dataset.csv"  # Replace with the actual path to your dataset
try:
    X_train, y_train, X_val, y_val, X_test, y_test, word_to_idx, tag_to_idx = load_and_preprocess_data(file_path)
except FileNotFoundError as e:
    print(f"Error: {e}")
    print(
        "Please download the 'ner_dataset.csv' file and place it in the same directory as this script, or provide the correct path to the file.")
    import sys

    sys.exit(1)  # Exit the program.

/tmp/ipykernel_119883/931062671.py:31: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method="ffill")  # Fill missing values


In [5]:
# Pad sequences to the maximum length in the training set
sequence_length = max(len(s) for s in X_train)
X_train = pad_sequences(X_train, maxlen=sequence_length, padding="post")
y_train = pad_sequences(y_train, maxlen=sequence_length, padding="post")
X_val = pad_sequences(X_val, maxlen=sequence_length, padding="post")
y_val = pad_sequences(y_val, maxlen=sequence_length, padding="post")
X_test = pad_sequences(X_test, maxlen=sequence_length, padding="post")
y_test = pad_sequences(y_test, maxlen=sequence_length, padding="post")

In [6]:

# Convert tag sequences to one-hot encoding
num_tags = len(tag_to_idx)
y_train = np.array([to_categorical(tags, num_classes=num_tags) for tags in y_train])
y_val = np.array([to_categorical(tags, num_classes=num_tags) for tags in y_val])
y_test = np.array([to_categorical(tags, num_classes=num_tags) for tags in y_test])


In [7]:
# Baseline Model Evaluation
print("\nBaseline Model: Most Frequent Tag")
idx_to_tag = {i: t for t, i in tag_to_idx.items()}
y_pred_baseline = baseline_model(X_test, y_test, tag_to_idx)
baseline_metrics = evaluate_model(y_test, y_pred_baseline, idx_to_tag)
print(f"Overall Precision: {baseline_metrics['overall_precision']:.4f}")
print(f"Overall Recall: {baseline_metrics['overall_recall']:.4f}")
print(f"Overall F1-score: {baseline_metrics['overall_f1']:.4f}")
print(f"Overall Accuracy: {baseline_metrics['overall_accuracy']:.4f}")
print("Classification Report:\n", baseline_metrics['classification_report'])



Baseline Model: Most Frequent Tag
Overall Precision: 0.0000
Overall Recall: 0.0000
Overall F1-score: 0.0000
Overall Accuracy: 0.0000
Classification Report:
 No data


In [8]:
# LSTM Model Training and Evaluation
print("\nLSTM Model Training and Evaluation")
vocab_size = len(word_to_idx)
embedding_dim = 128
lstm_units = 256
model = build_lstm_model(vocab_size, num_tags, embedding_dim, lstm_units, sequence_length)



LSTM Model Training and Evaluation


In [9]:

# Define early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [10]:
# Train the model
history = model.fit(X_train, y_train, batch_size=32, epochs=2, validation_data=(X_val, y_val),
                    callbacks=[early_stopping])

Epoch 1/2


2025-05-18 02:59:29.524961: W tensorflow/tsl/framework/cpu_allocator_impl.cc:82] Allocation of 258560640 exceeds 10% of free system memory.


1080/1080 [==============================] - 368s 339ms/step - loss: 0.1302 - accuracy: 0.9704 - val_loss: 0.0433 - val_accuracy: 0.9881
Epoch 2/2
1080/1080 [==============================] - 377s 349ms/step - loss: 0.0332 - accuracy: 0.9903 - val_loss: 0.0347 - val_accuracy: 0.9896


In [11]:
# Evaluate the model on the test set
y_pred_lstm = model.predict(X_test)
y_pred_lstm = np.argmax(y_pred_lstm, axis=-1)
lstm_metrics = evaluate_model(y_test, y_pred_lstm, idx_to_tag)
print(f"Overall Precision: {lstm_metrics['overall_precision']:.4f}")
print(f"Overall Recall: {lstm_metrics['overall_recall']:.4f}")
print(f"Overall F1-score: {lstm_metrics['overall_f1']:.4f}")
print(f"Overall Accuracy: {lstm_metrics['overall_accuracy']:.4f}")
print("Classification Report:\n", lstm_metrics['classification_report'])

300/300 [==============================] - 28s 91ms/step


ValueError: Found input variables with inconsistent numbers of samples: [788633, 208955]

In [ ]:
# Plot training history (loss and accuracy)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
